# Video Studio — `wan21-1-3b` on Colab (manual backup)

Kaggle is the studio's automated GPU provider. Use this notebook only when Kaggle has no GPU free.

1. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. Stay on the page.
2. When it finishes, the browser downloads **`vs-result-wan21-1-3b.zip`**. Give that file to Claude
   (upload it to the session or commit it to the branch); Claude runs `vs-colab import`.
3. The last cell releases the GPU so your Colab allowance is not wasted.

Colab rules: run it yourself in this notebook UI; no SSH, remote desktop or web UIs; one account;
no deepfakes — lip sync only on faces you own, AI-generated people, or people who agreed in writing.
Free Colab limits are not published and change; a run can be stopped if you are idle.

Same code as the Kaggle job (packed by `vs-colab build` from `gpu-jobs/`); models are downloaded
from their official Hugging Face / GitHub sources at pinned revisions.

In [ ]:
# 1. GPU check + map the Kaggle paths the job uses onto /content
import os, subprocess, pathlib
r = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True)
assert r.returncode == 0, 'No GPU: Runtime > Change runtime type > T4 GPU, then Run all again'
print(r.stdout)
for d in ('/content/kaggle/working', '/content/kaggle/temp'):
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)
if not os.path.exists('/kaggle'):
    os.symlink('/content/kaggle', '/kaggle')


In [ ]:
# 3. the studio job (generated — do not edit; rebuild with vs-colab build)
JOB = '# --- packed by vs-gpu: inputs, params, shared runtime (gpu-jobs/_common.py), then the job ---\nimport base64, json, os, pathlib, time\n_T0 = time.time()\nIN = pathlib.Path("/kaggle/working/in"); OUT = pathlib.Path("/kaggle/working/out")\nIN.mkdir(parents=True, exist_ok=True); OUT.mkdir(parents=True, exist_ok=True)\nPARAMS = json.loads(\'{"model": "wan21-1.3b", "preset": "t4", "items": [{"id": "kite", "prompt": "a red kite flying over green hills on a sunny day, clouds drifting, slow pan right, cinematic", "seed": 3}, {"id": "street", "prompt": "a young courier rides an orange scooter through a busy Cairo street at sunrise, tracking shot, cinematic", "seed": 7}], "item_budget_s": 1200, "max_job_s": 3600, "_need_gpu": true, "_need_internet": true, "parallel": false}\')\nfor _n, _b in json.loads(\'{}\').items():\n    (IN / _n).write_bytes(base64.b64decode(_b))\n_COMMON_B64 = \'IyBTaGFyZWQgcnVudGltZSBmb3IgZXZlcnkgc3R1ZGlvIEdQVSBqb2IgKHBhY2tlZCBieSB2cy1ncHU7IHJ1bnMgaW5zaWRlIHRoZSBLYWdnbGUga2VybmVsIGFuZCBpdHMKIyBwZXItR1BVIGNoaWxkIHByb2Nlc3NlcykuIFByb3ZpZGVzOiByZXBvcnQoKSwgZ3B1X2luZm8oKSwgaXNfdDQoKSwgcGlja19kdHlwZSgpLCBzaGFyZCgpLCBsb2FkX3BpcGUoKS4KIyBLYWdnbGUgZnJlZSB0aWVyID0gMnggVGVzbGEgVDQgKHNtNzUsIDE0LjYgR0IgZWFjaCk6IGZhc3QgZnAxNiB0ZW5zb3IgY29yZXMsIGJmMTYgb25seSBlbXVsYXRlZC4KaW1wb3J0IGpzb24sIG9zLCBwYXRobGliLCBzdWJwcm9jZXNzLCBzeXMsIHRpbWUKClNIQVJEID0gaW50KG9zLmVudmlyb24uZ2V0KCJWU19TSEFSRCIsICItMSIpKSAgICAgICAgICAjID49IDAgaW5zaWRlIGEgcGVyLUdQVSBjaGlsZCBwcm9jZXNzCgoKZGVmIHJlcG9ydCgqKmt3KToKICAgIHAgPSBPVVQgLyAoInJlcG9ydC5qc29uIiBpZiBTSEFSRCA8IDAgZWxzZSBmInJlcG9ydC5zaGFyZHtTSEFSRH0uanNvbiIpCiAgICBkID0ganNvbi5sb2FkcyhwLnJlYWRfdGV4dCgpKSBpZiBwLmV4aXN0cygpIGVsc2Uge30KICAgIGQudXBkYXRlKGt3KTsgZFsiZWxhcHNlZF9zIl0gPSByb3VuZCh0aW1lLnRpbWUoKSAtIF9UMCwgMSk7IHAud3JpdGVfdGV4dChqc29uLmR1bXBzKGQsIGluZGVudD0yKSkKCgpkZWYgX3ZyYW1fcGVhaygpOiAgICAgICAgICAgIyBiZW5jaG1hcms6IHBlYWsgVlJBTSBwZXIgdmlzaWJsZSBHUFUsIHdyaXR0ZW4gYXQgZXhpdCBvZiBldmVyeSAoY2hpbGQpIHByb2Nlc3MKICAgIHRyeToKICAgICAgICBpbXBvcnQgdG9yY2gKICAgICAgICBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOgogICAgICAgICAgICByZXBvcnQoKip7InZyYW1fcGVha19nYiI6IFtyb3VuZCh0b3JjaC5jdWRhLm1heF9tZW1vcnlfYWxsb2NhdGVkKGkpIC8gMioqMzAsIDIpCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGZvciBpIGluIHJhbmdlKHRvcmNoLmN1ZGEuZGV2aWNlX2NvdW50KCkpXX0pCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIHBhc3MKCgpkZWYgX3dhdGNoZG9nKGxpbWl0X3MpOiAgICAjIGhhcmQgam9iIGJ1ZGdldDogbmV2ZXIgbGV0IGEgc3R1Y2sgam9iIGJ1cm4gR1BVIHF1b3RhIChLYWdnbGUgYWxsb3dzIDEyIGgpCiAgICBpbXBvcnQgdGhyZWFkaW5nCiAgICBkZWYgZmlyZSgpOgogICAgICAgIHJlcG9ydChlcnJvcj1mImpvYiBleGNlZWRlZCBpdHMgYnVkZ2V0IG9mIHtsaW1pdF9zfSBzIGFuZCB3YXMgc3RvcHBlZCAocmFpc2UgbWF4X2pvYl9zIG9ubHkgaWYgbmVlZGVkKSIsCiAgICAgICAgICAgICAgIHRpbWVkX291dD1UcnVlKQogICAgICAgIF92cmFtX3BlYWsoKTsgb3MuX2V4aXQoNSkKICAgIHQgPSB0aHJlYWRpbmcuVGltZXIobGltaXRfcywgZmlyZSk7IHQuZGFlbW9uID0gVHJ1ZTsgdC5zdGFydCgpCgoKaW1wb3J0IGF0ZXhpdAphdGV4aXQucmVnaXN0ZXIoX3ZyYW1fcGVhaykKaWYgU0hBUkQgPCAwIGFuZCBub3Qgb3MuZW52aXJvbi5nZXQoIlZTX0ZBS0VfR1BVUyIpOgogICAgX3dhdGNoZG9nKGludChQQVJBTVMuZ2V0KCJtYXhfam9iX3MiLCAzNjAwKSkpCgoKZGVmIHN0ZXBfZ3VhcmQodG90YWxfc3RlcHMsIGJ1ZGdldF9zLCBsYWJlbD0iaXRlbSIpOgogICAgIiIiZGlmZnVzZXJzIGNhbGxiYWNrX29uX3N0ZXBfZW5kOiBzdG9wIG9uIE5hTi9pbmYgbGF0ZW50cyBhbmQgd2hlbiB0aGUgcHJvamVjdGVkIHRpbWUgb2YgdGhpcyBpdGVtIGV4Y2VlZHMKICAgIGJ1ZGdldF9zIChlYXJseSBmYWlsdXJlIGRldGVjdGlvbiwgbWVhc3VyZWQgcGVyIHN0ZXApLiBSYWlzZXMgRmxvYXRpbmdQb2ludEVycm9yIC8gVGltZW91dEVycm9yLiIiIgogICAgdDAgPSB0aW1lLnRpbWUoKQogICAgc3RhdHMgPSB7InN0ZXBzIjogMCwgImRlbm9pc2VfcyI6IDAuMH0KICAgIGRlZiBjYihwaXBlLCBzdGVwLCB0cywga3cpOgogICAgICAgIGltcG9ydCB0b3JjaAogICAgICAgIGxhdCA9IGt3LmdldCgibGF0ZW50cyIpCiAgICAgICAgaWYgbGF0IGlzIG5vdCBOb25lIGFuZCBub3QgdG9yY2guaXNmaW5pdGUobGF0KS5hbGwoKToKICAgICAgICAgICAgcmFpc2UgRmxvYXRpbmdQb2ludEVycm9yKGYie2xhYmVsfTogbm9uLWZpbml0ZSBsYXRlbnRzIGF0IHN0ZXAge3N0ZXB9IikKICAgICAgICBlbCA9IHRpbWUudGltZSgpIC0gdDAKICAgICAgICBpZiBzdGVwID49IDEgYW5kIGVsIC8gKHN0ZXAgKyAxKSAqIHRvdGFsX3N0ZXBzID4gYnVkZ2V0X3M6CiAgICAgICAgICAgIHJhaXNlIFRpbWVvdXRFcnJvcihmIntsYWJlbH06IHtlbCAvIChzdGVwICsgMSk6LjFmfSBzL3N0ZXAgeCB7dG90YWxfc3RlcHN9IHN0ZXBzID4gYnVkZ2V0IHtidWRnZXRfc30gcyIpCiAgICAgICAgc3RhdHMudXBkYXRlKHN0ZXBzPXN0ZXAgKyAxLCBkZW5vaXNlX3M9cm91bmQoZWwsIDEpKQogICAgICAgIHJldHVybiBrdwogICAgY2Iuc3RhdHMgPSBzdGF0cwogICAgcmV0dXJuIGNiCgoKZGVmIGdwdV9pbmZvKCk6CiAgICBpZiBvcy5lbnZpcm9uLmdldCgiVlNfRkFLRV9HUFVTIik6ICAgICAgICMgQ1BVIHNlbGYtdGVzdCBvZiBwYWNraW5nIGFuZCBmYW4tb3V0ICh0ZXN0cy90ZXN0X2dwdV9ydW50aW1lLnNoKQogICAgICAgIHJldHVybiB7Im4iOiBpbnQob3MuZW52aXJvblsiVlNfRkFLRV9HUFVTIl0pLCAibmFtZSI6ICJmYWtlIiwgInZyYW1fZ2IiOiAxNC42LCAiY2MiOiA3LjV9CiAgICBpbXBvcnQgdG9yY2gKICAgIG4gPSB0b3JjaC5jdWRhLmRldmljZV9jb3VudCgpCiAgICBpZiBub3QgbjoKICAgICAgICByZXR1cm4geyJuIjogMH0KICAgIHAgPSB0b3JjaC5jdWRhLmdldF9kZXZpY2VfcHJvcGVydGllcygwKQogICAgcmV0dXJuIHsibiI6IG4sICJuYW1lIjogcC5uYW1lLCAidnJhbV9nYiI6IHJvdW5kKHAudG90YWxfbWVtb3J5IC8gMioqMzAsIDEpLCAiY2MiOiBwLm1ham9yICsgcC5taW5vciAvIDEwfQoKCmRlZiBpc190NCgpOgogICAgZyA9IGdwdV9pbmZvKCkKICAgIHJldHVybiBnLmdldCgibiIsIDApID4gMCBhbmQgZ1siY2MiXSA8IDguMAoKCmRlZiBwaWNrX2R0eXBlKGZwMTZfc2FmZSk6CiAgICAiIiJmcDE2IG9uIFQ0IGZvciBtb2RlbHMga25vd24gdG8gYmUgZnAxNi1zYWZlICg0LTh4IGZhc3RlciB0aGFuIGVtdWxhdGVkIGJmMTYpOyBiZjE2IG90aGVyd2lzZS4KICAgIFBBUkFNU1siZHR5cGUiXSA9IGZwMTYgfCBiZjE2IHwgZnAzMiBvdmVycmlkZXMuIiIiCiAgICBpbXBvcnQgdG9yY2gKICAgIHdhbnQgPSBQQVJBTVMuZ2V0KCJkdHlwZSIsICJhdXRvIikKICAgIGlmIHdhbnQgaW4gKCJmcDE2IiwgImJmMTYiLCAiZnAzMiIpOgogICAgICAgIHJldHVybiB7ImZwMTYiOiB0b3JjaC5mbG9hdDE2LCAiYmYxNiI6IHRvcmNoLmJmbG9hdDE2LCAiZnAzMiI6IHRvcmNoLmZsb2F0MzJ9W3dhbnRdCiAgICBpZiBpc190NCgpOgogICAgICAgIHJldHVybiB0b3JjaC5mbG9hdDE2IGlmIGZwMTZfc2FmZSBlbHNlIHRvcmNoLmJmbG9hdDE2CiAgICByZXR1cm4gdG9yY2guYmZsb2F0MTYKCgpjbGFzcyBfRG9uZShFeGNlcHRpb24pOgogICAgcGFzcwoKCmRlZiBzaGFyZChpdGVtcyk6CiAgICAiIiJTbWFsbCBtb2RlbHM6IG9uZSBwcm9jZXNzIHBlciBHUFUsIGl0ZW1zIHNwbGl0IHJvdW5kLXJvYmluICgyeCB0aHJvdWdocHV0IG9uIDJ4IFQ0KS4KICAgIFJldHVybnMgdGhpcyBwcm9jZXNzJ3MgaXRlbXM7IGluIHRoZSBwYXJlbnQgaXQgd2FpdHMgZm9yIHRoZSBjaGlsZHJlbiwgbWVyZ2VzIHJlcG9ydHMgYW5kIHN0b3BzIHRoZSBqb2IuIiIiCiAgICBuID0gZ3B1X2luZm8oKS5nZXQoIm4iLCAwKQogICAgaWYgU0hBUkQgPj0gMDoKICAgICAgICByZXR1cm4gaXRlbXNbU0hBUkQ6OmludChvcy5lbnZpcm9uWyJWU19OU0hBUkQiXSldCiAgICBpZiBuIDwgMiBvciBsZW4oaXRlbXMpIDwgMiBvciBub3QgUEFSQU1TLmdldCgicGFyYWxsZWwiLCBUcnVlKToKICAgICAgICByZXR1cm4gaXRlbXMKICAgIHRtcCA9IHBhdGhsaWIuUGF0aCgiL2thZ2dsZS90ZW1wIik7IHRtcC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAodG1wIC8gInZzX3BhcmFtcy5qc29uIikud3JpdGVfdGV4dChqc29uLmR1bXBzKFBBUkFNUykpCiAgICAodG1wIC8gInZzX2NoaWxkLnB5Iikud3JpdGVfdGV4dCgKICAgICAgICAiaW1wb3J0IGJhc2U2NCwganNvbiwgcGF0aGxpYiwgdGltZVxuX1QwID0gdGltZS50aW1lKClcbiIKICAgICAgICAiSU4gPSBwYXRobGliLlBhdGgoJy9rYWdnbGUvd29ya2luZy9pbicpOyBPVVQgPSBwYXRobGliLlBhdGgoJy9rYWdnbGUvd29ya2luZy9vdXQnKVxuIgogICAgICAgICJQQVJBTVMgPSBqc29uLmxvYWQob3BlbignL2thZ2dsZS90ZW1wL3ZzX3BhcmFtcy5qc29uJykpXG4iCiAgICAgICAgZiJfQ09NTU9OX0I2NCA9IHtfQ09NTU9OX0I2NCFyfVxuX0pPQl9CNjQgPSB7X0pPQl9CNjQhcn1cbiIKICAgICAgICAiZXhlYyhiYXNlNjQuYjY0ZGVjb2RlKF9DT01NT05fQjY0KS5kZWNvZGUoKSlcbiIKICAgICAgICAiZXhlYyhjb21waWxlKGJhc2U2NC5iNjRkZWNvZGUoX0pPQl9CNjQpLmRlY29kZSgpLCAnam9iLnB5JywgJ2V4ZWMnKSlcbiIpCiAgICBwcm9jcyA9IFtzdWJwcm9jZXNzLlBvcGVuKFtzeXMuZXhlY3V0YWJsZSwgc3RyKHRtcCAvICJ2c19jaGlsZC5weSIpXSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZW52PXsqKm9zLmVudmlyb24sICJWU19TSEFSRCI6IHN0cihpKSwgIlZTX05TSEFSRCI6IHN0cihuKSwgIkNVREFfVklTSUJMRV9ERVZJQ0VTIjogc3RyKGkpfSkKICAgICAgICAgICAgIGZvciBpIGluIHJhbmdlKG4pXQogICAgY29kZXMgPSBbcC53YWl0KCkgZm9yIHAgaW4gcHJvY3NdCiAgICBzaGFyZHMgPSBbXQogICAgZm9yIGkgaW4gcmFuZ2Uobik6CiAgICAgICAgZiA9IE9VVCAvIGYicmVwb3J0LnNoYXJke2l9Lmpzb24iCiAgICAgICAgaWYgZi5leGlzdHMoKToKICAgICAgICAgICAgc2hhcmRzLmFwcGVuZChqc29uLmxvYWRzKGYucmVhZF90ZXh0KCkpKTsgZi51bmxpbmsoKQogICAgcmVwb3J0KHBhcmFsbGVsX2dwdXM9biwgc2hhcmRzPXNoYXJkcywgc2hhcmRfZXhpdF9jb2Rlcz1jb2RlcykKICAgIGlmIGNvZGVzIGFuZCBhbGwoY29kZXMpOgogICAgICAgIHJlcG9ydChlcnJvcj1mImV2ZXJ5IEdQVSB3b3JrZXIgZmFpbGVkOiBleGl0IGNvZGVzIHtjb2Rlc30iKQogICAgZWxpZiBhbnkoY29kZXMpOiAgICAgICAgICAgICAgICAgICAgICAgICAgIyBwYXJ0aWFsIHJlc3VsdDoga2VlcCB3aGF0IHdhcyBtYWRlLCB2cy1wcm9kdWNlIGZhbGxzIGJhY2sgZm9yIHRoZSByZXN0CiAgICAgICAgcmVwb3J0KHdhcm5pbmc9ZiJhIEdQVSB3b3JrZXIgZmFpbGVkOiBleGl0IGNvZGVzIHtjb2Rlc30iKQogICAgcmFpc2UgX0RvbmUoKQoKCmRlZiBsb2FkX3BpcGUoY2xzLCByZXBvLCByZXZpc2lvbiwgZHR5cGUsIGZwOF9vaz1UcnVlLCBwcmVmZXI9Tm9uZSwgKiprdyk6CiAgICAiIiJCaWcgZGlmZnVzaW9uIG1vZGVscyBvbiAyeCBUNCwgYmVzdCBzdHJhdGVneSBmaXJzdDoKICAgICAgIDEpIGRldmljZV9tYXA9J2JhbGFuY2VkJyDigJQgdGV4dCBlbmNvZGVyIG9uIG9uZSBUNCwgdHJhbnNmb3JtZXIgKyBWQUUgb24gdGhlIG90aGVyIChubyBDUFUgcm91bmQtdHJpcHMpCiAgICAgICAyKSBmcDggd2VpZ2h0IHN0b3JhZ2UgZm9yIHRoZSB0cmFuc2Zvcm1lciArIG1vZGVsIENQVSBvZmZsb2FkIG9uIG9uZSBUNAogICAgICAgMykgc2VxdWVudGlhbCBDUFUgb2ZmbG9hZCAoc2xvd2VzdCwgYWx3YXlzIGZpdHMpCiAgICBQQVJBTVNbInBsYWNlbWVudCJdID0gYmFsYW5jZWQgfCBvZmZsb2FkIHwgc2VxdWVudGlhbCBmb3JjZXMgb25lLiIiIgogICAgaW1wb3J0IHRvcmNoCiAgICBmb3JjZSA9IHByZWZlciBvciBQQVJBTVMuZ2V0KCJwbGFjZW1lbnQiKQogICAgZXJyb3JzID0gW10KICAgIGlmIChmb3JjZSBpbiAoTm9uZSwgImJhbGFuY2VkIikpIGFuZCBncHVfaW5mbygpLmdldCgibiIsIDApID49IDI6CiAgICAgICAgdHJ5OgogICAgICAgICAgICBwaXBlID0gY2xzLmZyb21fcHJldHJhaW5lZChyZXBvLCByZXZpc2lvbj1yZXZpc2lvbiwgdG9yY2hfZHR5cGU9ZHR5cGUsIGRldmljZV9tYXA9ImJhbGFuY2VkIiwgKiprdykKICAgICAgICAgICAgZnA4ID0gZnA4X29rIGFuZCBncHVfaW5mbygpWyJ2cmFtX2diIl0gPCAyMCBhbmQgaGFzYXR0cihwaXBlLCAidHJhbnNmb3JtZXIiKQogICAgICAgICAgICBpZiBmcDg6ICAgIyBtZWFzdXJlZDogWi1JbWFnZSdzIDZCIHRyYW5zZm9ybWVyIGFsb25lICgxMiBHQiBiZjE2KSArIGF0dGVudGlvbiBvdmVyZmxvd3MgYSAxNC42IEdCIFQ0CiAgICAgICAgICAgICAgICBwaXBlLnRyYW5zZm9ybWVyLmVuYWJsZV9sYXllcndpc2VfY2FzdGluZyhzdG9yYWdlX2R0eXBlPXRvcmNoLmZsb2F0OF9lNG0zZm4sIGNvbXB1dGVfZHR5cGU9ZHR5cGUpCiAgICAgICAgICAgICAgICB0b3JjaC5jdWRhLmVtcHR5X2NhY2hlKCkKICAgICAgICAgICAgcmVwb3J0KHBsYWNlbWVudD0iYmFsYW5jZWQgKDIgR1BVcykiICsgKCIgKyBmcDggdHJhbnNmb3JtZXIgd2VpZ2h0cyIgaWYgZnA4IGVsc2UgIiIpKQogICAgICAgICAgICByZXR1cm4gcGlwZQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICAgICAgZXJyb3JzLmFwcGVuZChmImJhbGFuY2VkOiB7dHlwZShlKS5fX25hbWVfX306IHtzdHIoZSlbOjIwMF19IikKICAgIGlmIGZvcmNlIGluIChOb25lLCAib2ZmbG9hZCIpOgogICAgICAgIHRyeToKICAgICAgICAgICAgcGlwZSA9IGNscy5mcm9tX3ByZXRyYWluZWQocmVwbywgcmV2aXNpb249cmV2aXNpb24sIHRvcmNoX2R0eXBlPWR0eXBlLCAqKmt3KQogICAgICAgICAgICBpZiBmcDhfb2sgYW5kIGdwdV9pbmZvKClbInZyYW1fZ2IiXSA8IDIwIGFuZCBoYXNhdHRyKHBpcGUsICJ0cmFuc2Zvcm1lciIpOgogICAgICAgICAgICAgICAgcGlwZS50cmFuc2Zvcm1lci5lbmFibGVfbGF5ZXJ3aXNlX2Nhc3Rpbmcoc3RvcmFnZV9kdHlwZT10b3JjaC5mbG9hdDhfZTRtM2ZuLCBjb21wdXRlX2R0eXBlPWR0eXBlKQogICAgICAgICAgICBwaXBlLmVuYWJsZV9tb2RlbF9jcHVfb2ZmbG9hZChncHVfaWQ9MCkKICAgICAgICAgICAgcmVwb3J0KHBsYWNlbWVudD0iZnA4IHRyYW5zZm9ybWVyICsgbW9kZWwgQ1BVIG9mZmxvYWQgKDEgR1BVKSIsIHBsYWNlbWVudF9lcnJvcnM9ZXJyb3JzKQogICAgICAgICAgICByZXR1cm4gcGlwZQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICAgICAgZXJyb3JzLmFwcGVuZChmIm9mZmxvYWQ6IHt0eXBlKGUpLl9fbmFtZV9ffToge3N0cihlKVs6MjAwXX0iKQogICAgcGlwZSA9IGNscy5mcm9tX3ByZXRyYWluZWQocmVwbywgcmV2aXNpb249cmV2aXNpb24sIHRvcmNoX2R0eXBlPWR0eXBlLCAqKmt3KQogICAgcGlwZS5lbmFibGVfc2VxdWVudGlhbF9jcHVfb2ZmbG9hZCgpCiAgICByZXBvcnQocGxhY2VtZW50PSJzZXF1ZW50aWFsIENQVSBvZmZsb2FkIiwgcGxhY2VtZW50X2Vycm9ycz1lcnJvcnMpCiAgICByZXR1cm4gcGlwZQoKCmRlZiB2YWVfdGlsaW5nKHBpcGUpOgogICAgZm9yIG5hbWUgaW4gKCJ2YWUiLCk6CiAgICAgICAgbSA9IGdldGF0dHIocGlwZSwgbmFtZSwgTm9uZSkKICAgICAgICBmb3IgZm4gaW4gKCJlbmFibGVfdGlsaW5nIiwgImVuYWJsZV9zbGljaW5nIik6CiAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgIGdldGF0dHIobSwgZm4pKCkKICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgICAgIHBhc3MKCgpkZWYgbG9va3NfYnJva2VuKGFycik6CiAgICAiIiJOYU4vYmxhY2sgb3V0cHV0IGNoZWNrIChmcDE2IG92ZXJmbG93IHNob3dzIHVwIGFzIGJsYWNrIGZyYW1lcykuIiIiCiAgICBpbXBvcnQgbnVtcHkgYXMgbnAKICAgIGEgPSBucC5hc2FycmF5KGFyciwgZHR5cGU9bnAuZmxvYXQzMikKICAgIHJldHVybiAobm90IG5wLmlzZmluaXRlKGEpLmFsbCgpKSBvciBmbG9hdChhLm1lYW4oKSkgPCAyLjAK\'\n_JOB_B64 = \'IyBHUFUgam9iOiB0ZXh0LXRvLXZpZGVvIGFuZCBpbWFnZS10by12aWRlbywgdHVuZWQgZm9yIEthZ2dsZSdzIDJ4IFRlc2xhIFQ0ICgxNC42IEdCIGVhY2gsIHNtNzUpLgojIFBBUkFNUzogeyJtb2RlbCI6ICJ3YW4yMS0xLjNiIiB8ICJ3YW4yMi01YiIsICJwcmVzZXQiOiAidDQiIHwgInQ0LWZhc3QiIHwgImhxIiwgImFsbG93X2hlYXZ5IjogZmFsc2UsCiMgICAgICAgICAgIml0ZW1fYnVkZ2V0X3MiOiAxNTAwLCAibWF4X2pvYl9zIjogNTQwMCwKIyAgICAgICAgICAiaXRlbXMiOiBbeyJpZCIsInByb21wdCIsImltYWdlIj8gKGtleWZyYW1lIGluIElOLyAtPiBpbWFnZS10by12aWRlbyksImZyYW1lcyIsIndpZHRoIiwiaGVpZ2h0Iiwic2VlZCJ9XSwKIyAgICAgICAgICAic3RlcHMiOiBOLCAiZnBzIjogMjQsICJkdHlwZSI6ICJhdXRvInwiZnAxNiJ8ImJmMTYifQojCiMgICB3YW4yMS0xLjNiIERFRkFVTFQgb24gVDQuIFdhbjIuMS1UMlYtMS4zQiAoQXBhY2hlLTIuMCk6IGxpZ2h0IHRleHQtdG8tdmlkZW8gYXQgNDgwcDsgdGV4dC1vbmx5LgojICAgd2FuMjItNWIgICBBRFZBTkNFRCAob3B0aW9uYWwsIGFsbG93X2hlYXZ5KS4gV2FuMi4yLVRJMlYtNUIgKEFwYWNoZS0yLjApOiB0ZXh0LSBBTkQgaW1hZ2UtdG8tdmlkZW8uCiMgICAgICAgICAgICAgIE1lYXN1cmVkIG9uIFQ0IChiYWxhbmNlZCArIGZwOCk6IDcuOCBzL3N0ZXAsIFZBRSBkZWNvZGUgb2ZmLUdQVSB+NjIgbWluLCBibGFjayBmcDE2IG91dHB1dC4KIyAgIEJvdGg6IG9uZSBUNCwgbW9kZWwgQ1BVIG9mZmxvYWQgKFVNVDUgZW5jb2RlcyB0aGVuIHN3YXBzIG91dCksIGZwMTYgY29tcHV0ZSAoVDQgaGFzIG5vIG5hdGl2ZSBiZjE2KSwKIyAgIFZBRSB0aWxpbmcsIHBlci1zdGVwIGd1YXJkIChOYU4gbGF0ZW50cyBvciBwcm9qZWN0ZWQgdGltZSA+IGl0ZW1fYnVkZ2V0X3Mgc3RvcHMgdGhlIGNsaXAgZWFybHkpLAojICAgY2xpcHMgd2l0aCBOYU4gYXJlIHJlZG9uZSBvbmNlIGluIGJmMTY7IGhhcmQgam9iIGJ1ZGdldCBtYXhfam9iX3MuCiMgICBMb3cgaG9zdCBSQU0gKDwgMjAgR0IsIGUuZy4gZnJlZSBDb2xhYidzIDEyLjcgR0I7IG9yIFBBUkFNUyBsb3dfcmFtKTogdGhlIDExIEdCIFVNVDUgdGV4dCBlbmNvZGVyIHdvdWxkIG5vdAojICAgZml0IGluIFJBTSBuZXh0IHRvIHRoZSB0cmFuc2Zvcm1lciBmb3IgQ1BVIG9mZmxvYWQsIHNvIGFsbCBwcm9tcHRzIGFyZSBlbmNvZGVkIGZpcnN0IHdpdGggdGhlIGVuY29kZXIgb24gdGhlCiMgICBHUFUsIHRoZSBlbmNvZGVyIGlzIGZyZWVkLCBhbmQgdGhlIHBpcGVsaW5lIGlzIGJ1aWx0IHdpdGhvdXQgaXQgKHByb21wdCBlbWJlZGRpbmdzIHBhc3NlZCBpbikuCiMKIyBQcmVzZXRzIChwZXItaXRlbSB3aWR0aC9oZWlnaHQvZnJhbWVzIG92ZXJyaWRlIHRoZW0pOgojICAgdDQgICAgICAgODMyeDQ4MCwgODEgZnJhbWVzICgzLjQgcyBAMjQpLCAyMCBzdGVwcyAg4oCUIGRlZmF1bHQgb24gVDQ7IHZzLXByb2R1Y2UgdXBzY2FsZXMgdG8gdGhlIGRlbGl2ZXJ5IHNpemUKIyAgIHQ0LWZhc3QgIDgzMng0ODAsIDQ5IGZyYW1lcyAoMi4wIHMpLCAxNiBzdGVwcyAgICAgIOKAlCBkcmFmdHMgYW5kIHNob3J0IGN1dGF3YXlzCiMgICBocSAgICAgICAxMjgweDcwNCwgMTIxIGZyYW1lcyAoNSBzKSwgMzAgc3RlcHMgICAgICDigJQgb25seSBvbiBiaWdnZXIgR1BVcyAoTDQvQTEwMCk7IGhvdXJzIHBlciBjbGlwIG9uIFQ0CmltcG9ydCBvcywgc3VicHJvY2Vzcywgc3lzCm9zLmVudmlyb24uc2V0ZGVmYXVsdCgiUFlUT1JDSF9DVURBX0FMTE9DX0NPTkYiLCAiZXhwYW5kYWJsZV9zZWdtZW50czpUcnVlIikKc3VicHJvY2Vzcy5ydW4oW3N5cy5leGVjdXRhYmxlLCAiLW0iLCAicGlwIiwgImluc3RhbGwiLCAiLXEiLCAiZGlmZnVzZXJzPT0wLjQzLjAiLCAiYWNjZWxlcmF0ZSIsICJ0cmFuc2Zvcm1lcnMiLCAiZnRmeSIsCiAgICAgICAgICAgICAgICAiaW1hZ2Vpby1mZm1wZWciXSwgY2hlY2s9VHJ1ZSkKaW1wb3J0IHRvcmNoCmZyb20gZGlmZnVzZXJzLnV0aWxzIGltcG9ydCBleHBvcnRfdG9fdmlkZW8sIGxvYWRfaW1hZ2UKClBSRVNFVFMgPSB7InQ0IjogKDgzMiwgNDgwLCA4MSwgMjApLCAidDQtZmFzdCI6ICg4MzIsIDQ4MCwgNDksIDE2KSwgImhxIjogKDEyODAsIDcwNCwgMTIxLCAzMCl9ClJFUE9TID0geyJ3YW4yMi01YiI6ICgiV2FuLUFJL1dhbjIuMi1USTJWLTVCLURpZmZ1c2VycyIsICJiOGZmZjczMTVjNzYiKSwKICAgICAgICAgIndhbjIxLTEuM2IiOiAoIldhbi1BSS9XYW4yLjEtVDJWLTEuM0ItRGlmZnVzZXJzIiwgIjBmYWQ3ODBhNTM0YiIpfQptb2RlbCA9IFBBUkFNUy5nZXQoIm1vZGVsIikgb3IgKCJ3YW4yMS0xLjNiIiBpZiBpc190NCgpIGVsc2UgIndhbjIyLTViIikKcHJlc2V0ID0gUEFSQU1TLmdldCgicHJlc2V0IiwgInQ0IiBpZiBpc190NCgpIGVsc2UgImhxIikKUFcsIFBILCBQRiwgUFMgPSBQUkVTRVRTW3ByZXNldF0KTkVHID0gImJsdXJyeSwgbG93IHF1YWxpdHksIGRpc3RvcnRlZCBmYWNlLCBleHRyYSBsaW1icywgd2F0ZXJtYXJrLCB0ZXh0LCBzdWJ0aXRsZXMsIHN0YXRpYyBmcmFtZSwgZmxpY2tlciIKaWYgbW9kZWwgPT0gIndhbjIxLTEuM2IiIGFuZCBhbnkoaXQuZ2V0KCJpbWFnZSIpIGZvciBpdCBpbiBQQVJBTVNbIml0ZW1zIl0pOgogICAgaWYgUEFSQU1TLmdldCgiYWxsb3dfaGVhdnkiKToKICAgICAgICBtb2RlbCA9ICJ3YW4yMi01YiIgICAgICAgICAgICAgICAgICAgIyBvbmx5IHRoZSA1QiBtb2RlbCBjYW4gYW5pbWF0ZSBhIGtleWZyYW1lCiAgICAgICAgcmVwb3J0KG5vdGU9ImltYWdlLXRvLXZpZGVvIHJlcXVlc3RlZDogdXNpbmcgd2FuMjItNWIgKGFsbG93X2hlYXZ5KSIpCiAgICBlbHNlOgogICAgICAgIHJlcG9ydChub3RlPSJ3YW4yMS0xLjNiIGlzIHRleHQtb25seToga2V5ZnJhbWVzIGlnbm9yZWQgKHNldCBhbGxvd19oZWF2eSBmb3Igd2FuMjItNWIgaW1hZ2UtdG8tdmlkZW8pIikKCgpkZWYgaG9zdF9yYW1fZ2IoKToKICAgIHRyeToKICAgICAgICByZXR1cm4gb3Muc3lzY29uZigiU0NfUEFHRV9TSVpFIikgKiBvcy5zeXNjb25mKCJTQ19QSFlTX1BBR0VTIikgLyAyKiozMAogICAgZXhjZXB0IChWYWx1ZUVycm9yLCBPU0Vycm9yKToKICAgICAgICByZXR1cm4gNjQuMAoKCkxPV19SQU0gPSBib29sKFBBUkFNUy5nZXQoImxvd19yYW0iLCBob3N0X3JhbV9nYigpIDwgMjApKQpFTUIgPSB7fQoKCmRlZiBwcmVfZW5jb2RlKGR0eXBlKToKICAgICIiIkVuY29kZSBldmVyeSBwcm9tcHQgKyBuZWdhdGl2ZSB3aXRoIFVNVDUgb24gdGhlIEdQVSwgdGhlbiBmcmVlIGl0IChsb3ctUkFNIGhvc3RzKS4iIiIKICAgIGZyb20gZGlmZnVzZXJzIGltcG9ydCBXYW5QaXBlbGluZQogICAgZnJvbSB0cmFuc2Zvcm1lcnMgaW1wb3J0IFVNVDVFbmNvZGVyTW9kZWwKICAgIHJlcG8sIHJldiA9IFJFUE9TW21vZGVsXQogICAgdGVfZHR5cGUgPSB0b3JjaC5iZmxvYXQxNiBpZiB0b3JjaC5jdWRhLmlzX2JmMTZfc3VwcG9ydGVkKGluY2x1ZGluZ19lbXVsYXRpb249VHJ1ZSkgZWxzZSBkdHlwZQogICAgdGUgPSBVTVQ1RW5jb2Rlck1vZGVsLmZyb21fcHJldHJhaW5lZChyZXBvLCBzdWJmb2xkZXI9InRleHRfZW5jb2RlciIsIHJldmlzaW9uPXJldiwgdG9yY2hfZHR5cGU9dGVfZHR5cGUsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGRldmljZV9tYXA9ImN1ZGEiLCBsb3dfY3B1X21lbV91c2FnZT1UcnVlKQogICAgZW5jID0gV2FuUGlwZWxpbmUuZnJvbV9wcmV0cmFpbmVkKHJlcG8sIHJldmlzaW9uPXJldiwgdGV4dF9lbmNvZGVyPXRlLCB0cmFuc2Zvcm1lcj1Ob25lLCB2YWU9Tm9uZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICB0b3JjaF9kdHlwZT10ZV9kdHlwZSkKICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgIGZvciBpdCBpbiBQQVJBTVNbIml0ZW1zIl06CiAgICAgICAgICAgIHBlLCBuZSA9IGVuYy5lbmNvZGVfcHJvbXB0KHByb21wdD1pdFsicHJvbXB0Il0sIG5lZ2F0aXZlX3Byb21wdD1pdC5nZXQoIm5lZ2F0aXZlIiwgTkVHKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZG9fY2xhc3NpZmllcl9mcmVlX2d1aWRhbmNlPVRydWUsIG51bV92aWRlb3NfcGVyX3Byb21wdD0xLCBkZXZpY2U9ImN1ZGEiLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBtYXhfc2VxdWVuY2VfbGVuZ3RoPTUxMikgICAjID0gV2FuUGlwZWxpbmUuX19jYWxsX18gZGVmYXVsdAogICAgICAgICAgICBpZiBub3QgKHRvcmNoLmlzZmluaXRlKHBlKS5hbGwoKSBhbmQgdG9yY2guaXNmaW5pdGUobmUpLmFsbCgpKToKICAgICAgICAgICAgICAgIHJhaXNlIEZsb2F0aW5nUG9pbnRFcnJvcihmIntpdFsnaWQnXX06IG5vbi1maW5pdGUgdGV4dCBlbWJlZGRpbmdzIikKICAgICAgICAgICAgRU1CW2l0WyJpZCJdXSA9IChwZS50bygiY3B1IiksIG5lLnRvKCJjcHUiKSkKICAgIGRlbCBlbmMsIHRlCiAgICBpbXBvcnQgZ2M7IGdjLmNvbGxlY3QoKTsgdG9yY2guY3VkYS5lbXB0eV9jYWNoZSgpCgoKZGVmIGJ1aWxkKGR0eXBlKToKICAgIGZyb20gZGlmZnVzZXJzIGltcG9ydCBXYW5QaXBlbGluZSwgV2FuSW1hZ2VUb1ZpZGVvUGlwZWxpbmUsIEF1dG9lbmNvZGVyS0xXYW4KICAgIHJlcG8sIHJldiA9IFJFUE9TW21vZGVsXQogICAgaWYgTE9XX1JBTToKICAgICAgICBpZiBub3QgRU1COgogICAgICAgICAgICBwcmVfZW5jb2RlKGR0eXBlKQogICAgICAgIHQydiA9IGxvYWRfcGlwZShXYW5QaXBlbGluZSwgcmVwbywgcmV2LCBkdHlwZSwgZnA4X29rPUZhbHNlLCBwcmVmZXI9UEFSQU1TLmdldCgicGxhY2VtZW50IiwgIm9mZmxvYWQiKSwKICAgICAgICAgICAgICAgICAgICAgICAgdmFlPUF1dG9lbmNvZGVyS0xXYW4uZnJvbV9wcmV0cmFpbmVkKHJlcG8sIHN1YmZvbGRlcj0idmFlIiwgcmV2aXNpb249cmV2LAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgdG9yY2hfZHR5cGU9dG9yY2guZmxvYXQzMiksIHRleHRfZW5jb2Rlcj1Ob25lKQogICAgICAgIHZhZV90aWxpbmcodDJ2KQogICAgICAgIHJldHVybiB0MnYsIChXYW5JbWFnZVRvVmlkZW9QaXBlbGluZSgqKnQydi5jb21wb25lbnRzKSBpZiBtb2RlbCA9PSAid2FuMjItNWIiIGVsc2UgTm9uZSkKICAgIHZhZSA9IEF1dG9lbmNvZGVyS0xXYW4uZnJvbV9wcmV0cmFpbmVkKHJlcG8sIHN1YmZvbGRlcj0idmFlIiwgcmV2aXNpb249cmV2LCB0b3JjaF9kdHlwZT10b3JjaC5mbG9hdDMyKQogICAgIyBNZWFzdXJlZCBvbiAyeCBUNCAoYmFsYW5jZWQgKyBmcDgpOiBkZW5vaXNlIDcuOCBzL3N0ZXAsIGJ1dCB0aGUgVkFFIGxhbmRlZCBvZmYtR1BVIChkZWNvZGUgfjYyIG1pbikgYW5kCiAgICAjIGZwMTYgb3V0cHV0IHdhcyBibGFjay4gU286IG9uZSBUNCB3aXRoIG1vZGVsIENQVSBvZmZsb2FkIChVTVQ1IGVuY29kZXMsIHRoZW4gc3dhcHMgb3V0OyB0aGUgNUIgdHJhbnNmb3JtZXIKICAgICMgZml0cyBpbiBmcDE2KSwgVkFFIG9uIHRoZSBHUFUsIG5vIGZwOCAoZnA4IGlzIG9ubHkgbmVlZGVkIGZvciA+MTAgR0IgdHJhbnNmb3JtZXJzKS4KICAgIHQydiA9IGxvYWRfcGlwZShXYW5QaXBlbGluZSwgcmVwbywgcmV2LCBkdHlwZSwgZnA4X29rPUZhbHNlLCBwcmVmZXI9UEFSQU1TLmdldCgicGxhY2VtZW50IiwgIm9mZmxvYWQiKSwgdmFlPXZhZSkKICAgIHZhZV90aWxpbmcodDJ2KQogICAgaTJ2ID0gV2FuSW1hZ2VUb1ZpZGVvUGlwZWxpbmUoKip0MnYuY29tcG9uZW50cykgaWYgbW9kZWwgPT0gIndhbjIyLTViIiBlbHNlIE5vbmUKICAgIHJldHVybiB0MnYsIGkydgoKCmRlZiBzbmFwKHYsIG0pOgogICAgcmV0dXJuIG1heChtLCBpbnQocm91bmQodiAvIG0pKSAqIG0pCgoKTEFTVCA9IHt9CgoKZGVmIHJlbmRlcihwaXBlcywgaXQsIGR0eXBlKToKICAgIHQydiwgaTJ2ID0gcGlwZXMKICAgIHcsIGggPSBpdC5nZXQoIndpZHRoIiwgUFcpLCBpdC5nZXQoImhlaWdodCIsIFBIKQogICAgaWYgaCA+IHcgYW5kICh3LCBoKSA9PSAoUFcsIFBIKToKICAgICAgICB3LCBoID0gaCwgdyAgICAgICAgICAgICAgICAgICAgICAgICAgIyB2ZXJ0aWNhbAogICAgaWYgIndpZHRoIiBub3QgaW4gaXQgYW5kICJoZWlnaHQiIG5vdCBpbiBpdCBhbmQgaXQuZ2V0KCJ2ZXJ0aWNhbCIpOgogICAgICAgIHcsIGggPSBQSCwgUFcKICAgIHcsIGggPSBzbmFwKHcsIDMyKSwgc25hcChoLCAzMikKICAgIGYgPSBpdC5nZXQoImZyYW1lcyIsIFBGKTsgZiA9IChmIC0gMSkgLy8gNCAqIDQgKyAxCiAgICBpZiBMT1dfUkFNOiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICMgZW1iZWRkaW5ncyBtYWRlIGJ5IHByZV9lbmNvZGUgKHRleHQgZW5jb2RlciBhbHJlYWR5IGZyZWVkKQogICAgICAgIHBlLCBuZSA9IEVNQltpdFsiaWQiXV0KICAgICAgICB0eHQgPSBkaWN0KHByb21wdF9lbWJlZHM9cGUudG8oImN1ZGEiLCBkdHlwZSksIG5lZ2F0aXZlX3Byb21wdF9lbWJlZHM9bmUudG8oImN1ZGEiLCBkdHlwZSkpCiAgICBlbHNlOgogICAgICAgIHR4dCA9IGRpY3QocHJvbXB0PWl0WyJwcm9tcHQiXSwgbmVnYXRpdmVfcHJvbXB0PWl0LmdldCgibmVnYXRpdmUiLCBORUcpKQogICAga3cgPSBkaWN0KCoqdHh0LCBoZWlnaHQ9aCwgd2lkdGg9dywgbnVtX2ZyYW1lcz1mLAogICAgICAgICAgICAgIGd1aWRhbmNlX3NjYWxlPWl0LmdldCgiY2ZnIiwgNS4wKSwgbnVtX2luZmVyZW5jZV9zdGVwcz1QQVJBTVMuZ2V0KCJzdGVwcyIsIFBTKSwKICAgICAgICAgICAgICBnZW5lcmF0b3I9dG9yY2guR2VuZXJhdG9yKCJjcHUiKS5tYW51YWxfc2VlZChpdC5nZXQoInNlZWQiLCAwKSkpCiAgICBnID0gc3RlcF9ndWFyZChrd1sibnVtX2luZmVyZW5jZV9zdGVwcyJdLCBQQVJBTVMuZ2V0KCJpdGVtX2J1ZGdldF9zIiwgMTUwMCksIGl0WyJpZCJdKQogICAga3dbImNhbGxiYWNrX29uX3N0ZXBfZW5kIl0gPSBnOyBMQVNUWyJndWFyZCJdID0gZwogICAgaWYgaXQuZ2V0KCJpbWFnZSIpIGFuZCBpMnYgaXMgbm90IE5vbmUgYW5kIG1vZGVsID09ICJ3YW4yMi01YiI6CiAgICAgICAgZnJhbWVzID0gaTJ2KGltYWdlPWxvYWRfaW1hZ2Uoc3RyKElOIC8gaXRbImltYWdlIl0pKS5yZXNpemUoKHcsIGgpKSwgKiprdykuZnJhbWVzWzBdCiAgICBlbHNlOgogICAgICAgIGZyYW1lcyA9IHQydigqKmt3KS5mcmFtZXNbMF0KICAgIHJldHVybiBmcmFtZXMsICh3LCBoLCBmKQoKCmRlZiB0aW1pbmcodG90YWwpOgogICAgc3QgPSBMQVNUWyJndWFyZCJdLnN0YXRzCiAgICByZXR1cm4geyJkZW5vaXNlX3MiOiBzdFsiZGVub2lzZV9zIl0sICJzX3Blcl9zdGVwIjogcm91bmQoc3RbImRlbm9pc2VfcyJdIC8gbWF4KDEsIHN0WyJzdGVwcyJdKSwgMiksCiAgICAgICAgICAgICJkZWNvZGVfcyI6IHJvdW5kKHRvdGFsIC0gc3RbImRlbm9pc2VfcyJdLCAxKX0KCgp0ID0gdGltZS50aW1lKCkKZHR5cGUgPSBwaWNrX2R0eXBlKGZwMTZfc2FmZT1UcnVlKQpwaXBlcyA9IGJ1aWxkKGR0eXBlKQpyZXBvcnQobG9hZF9zPXJvdW5kKHRpbWUudGltZSgpIC0gdCwgMSksIGR0eXBlPXN0cihkdHlwZSksIG1vZGVsPW1vZGVsLCBwcmVzZXQ9cHJlc2V0LCBsb3dfcmFtPUxPV19SQU0sCiAgICAgICBob3N0X3JhbV9nYj1yb3VuZChob3N0X3JhbV9nYigpLCAxKSkKZG9uZSwgcmVkbywgZmFpbGVkID0gW10sIFtdLCBbXQpmb3IgaXQgaW4gUEFSQU1TWyJpdGVtcyJdOgogICAgcyA9IHRpbWUudGltZSgpCiAgICB0cnk6CiAgICAgICAgZnJhbWVzLCBkaW1zID0gcmVuZGVyKHBpcGVzLCBpdCwgZHR5cGUpCiAgICBleGNlcHQgRmxvYXRpbmdQb2ludEVycm9yIGFzIGU6CiAgICAgICAgcmVkby5hcHBlbmQoaXQpOyByZXBvcnQoY2xpcHM9ZG9uZSwgYnJva2VuX2ZwMTY9W3JbImlkIl0gZm9yIHIgaW4gcmVkb10sIG5hbj1zdHIoZSkpOyBjb250aW51ZQogICAgZXhjZXB0IFRpbWVvdXRFcnJvciBhcyBlOiAgICAgICAgICAgICAgICAjIHRvbyBzbG93IGZvciB0aGlzIGhhcmR3YXJlOiBnaXZlIHVwIG9uIHRoaXMgY2xpcCwga2VlcCBxdW90YQogICAgICAgIGZhaWxlZC5hcHBlbmQoeyJpZCI6IGl0WyJpZCJdLCAid2h5Ijogc3RyKGUpfSk7IHJlcG9ydChjbGlwcz1kb25lLCBmYWlsZWQ9ZmFpbGVkKTsgY29udGludWUKICAgIGlmIGxvb2tzX2Jyb2tlbihmcmFtZXNbbGVuKGZyYW1lcykgLy8gMl0pOgogICAgICAgIHJlZG8uYXBwZW5kKGl0KTsgcmVwb3J0KGNsaXBzPWRvbmUsIGJyb2tlbl9mcDE2PVtyWyJpZCJdIGZvciByIGluIHJlZG9dKTsgY29udGludWUKICAgIGV4cG9ydF90b192aWRlbyhmcmFtZXMsIHN0cihPVVQgLyBmIntpdFsnaWQnXX0ubXA0IiksIGZwcz1QQVJBTVMuZ2V0KCJmcHMiLCAyNCkpCiAgICBkb25lLmFwcGVuZCh7ImlkIjogaXRbImlkIl0sICJzIjogcm91bmQodGltZS50aW1lKCkgLSBzLCAxKSwgInNpemUiOiBmIntkaW1zWzBdfXh7ZGltc1sxXX0iLCAiZnJhbWVzIjogZGltc1syXSwKICAgICAgICAgICAgICAgICAqKnRpbWluZyh0aW1lLnRpbWUoKSAtIHMpfSkKICAgIHJlcG9ydChjbGlwcz1kb25lKQppZiByZWRvIGFuZCBkdHlwZSA9PSB0b3JjaC5mbG9hdDE2OiAgICAgICAgICAjIGZwMTYgb3ZlcmZsb3dlZCBvbiB0aGVzZSBjbGlwczogcmVidWlsZCBvbmNlIGluIGJmMTYgKHNsb3dlciwgZXhhY3QpCiAgICBmb3IgcCBpbiBwaXBlczoKICAgICAgICBpZiBwIGlzIG5vdCBOb25lOgogICAgICAgICAgICB0cnk6IHAucmVtb3ZlX2FsbF9ob29rcygpCiAgICAgICAgICAgIGV4Y2VwdCBFeGNlcHRpb246IHBhc3MKICAgIGRlbCBwaXBlcywgcAogICAgaW1wb3J0IGdjOyBnYy5jb2xsZWN0KCk7IHRvcmNoLmN1ZGEuZW1wdHlfY2FjaGUoKQogICAgdCA9IHRpbWUudGltZSgpOyBwaXBlcyA9IGJ1aWxkKHRvcmNoLmJmbG9hdDE2KTsgcmVwb3J0KGJmMTZfbG9hZF9zPXJvdW5kKHRpbWUudGltZSgpIC0gdCwgMSkpCiAgICBmb3IgaXQgaW4gcmVkbzoKICAgICAgICBzID0gdGltZS50aW1lKCkKICAgICAgICB0cnk6CiAgICAgICAgICAgIGZyYW1lcywgZGltcyA9IHJlbmRlcihwaXBlcywgaXQsIHRvcmNoLmJmbG9hdDE2KQogICAgICAgIGV4Y2VwdCAoRmxvYXRpbmdQb2ludEVycm9yLCBUaW1lb3V0RXJyb3IpIGFzIGU6CiAgICAgICAgICAgIGZhaWxlZC5hcHBlbmQoeyJpZCI6IGl0WyJpZCJdLCAid2h5IjogImJmMTY6ICIgKyBzdHIoZSl9KTsgcmVwb3J0KGZhaWxlZD1mYWlsZWQpOyBjb250aW51ZQogICAgICAgIGV4cG9ydF90b192aWRlbyhmcmFtZXMsIHN0cihPVVQgLyBmIntpdFsnaWQnXX0ubXA0IiksIGZwcz1QQVJBTVMuZ2V0KCJmcHMiLCAyNCkpCiAgICAgICAgZG9uZS5hcHBlbmQoeyJpZCI6IGl0WyJpZCJdLCAicyI6IHJvdW5kKHRpbWUudGltZSgpIC0gcywgMSksICJkdHlwZSI6ICJiZjE2LXJldHJ5IiwgKip0aW1pbmcodGltZS50aW1lKCkgLSBzKX0pOyByZXBvcnQoY2xpcHM9ZG9uZSkKaWYgbm90IGRvbmU6CiAgICByZXBvcnQoZXJyb3I9Im5vIGNsaXAgd2FzIHByb2R1Y2VkOiAiICsgIjsgIi5qb2luKGYie2ZbJ2lkJ119OiB7Zlsnd2h5J119IiBmb3IgZiBpbiBmYWlsZWQpKQo=\'\nexec(base64.b64decode(_COMMON_B64).decode())\n\n\ndef _env_check():  # did Kaggle grant what the job asked for? (GPU + internet need a phone-verified account)\n    import shutil as _sh, socket as _so, subprocess as _sp\n    gpu = bool(_sh.which("nvidia-smi")) and _sp.run(["nvidia-smi", "-L"], capture_output=True).returncode == 0\n    try:\n        _so.getaddrinfo("huggingface.co", 443); net = True\n    except OSError:\n        net = False\n    report(gpu_granted=gpu, internet_granted=net)\n    if PARAMS.get("_need_gpu") and not gpu:\n        report(error=("Kaggle gave this session no GPU: this accelerator is busy/unavailable or another GPU session "\n                      "is running (free accounts get one at a time); retry later.") if net else\n                     ("Kaggle gave this session no GPU and no internet: the account needs phone verification "\n                      "(kaggle.com > Settings > Phone verification)."), gpu_unavailable_reason="busy" if net else "unverified")\n        raise SystemExit(3)\n    if PARAMS.get("_need_internet") and not net:\n        report(error="Kaggle gave this session no internet (phone verification needed, or internet off).")\n        raise SystemExit(3)\n\n\n_env_check()\ntry:\n    import torch as _t\n    if _t.cuda.is_available():\n        report(gpu=gpu_info())\nexcept Exception:\n    pass\ntry:\n    exec(compile(base64.b64decode(_JOB_B64).decode(), "job.py", "exec"))\nexcept _Done:\n    pass\nif json.loads((OUT / "report.json").read_text()).get("error"):\n    raise SystemExit(4)\n'
open('/content/vs_job.py', 'w').write(JOB)
import subprocess, sys
rc = subprocess.run([sys.executable, '/content/vs_job.py']).returncode
print('job exit code', rc, '(0 = ok; see report.json in the zip either way)')


In [ ]:
# 4. pack the results and download them
import json, pathlib, shutil
out = pathlib.Path('/kaggle/working/out')
rep = out / 'report.json'
if rep.exists():
    d = json.loads(rep.read_text()); d['provider'] = 'colab'; rep.write_text(json.dumps(d, indent=2))
    print(json.dumps(d, indent=1)[:3000])
zp = shutil.make_archive('/content/vs-result-wan21-1-3b', 'zip', out)
from google.colab import files
files.download(zp)


In [ ]:
# 5. release the GPU (run after the download has started)
from google.colab import runtime
runtime.unassign()
